In [8]:
import pandas as pd
import re

# Read the data from the CSV file (using absolute path)
df = pd.read_csv('/home/sheikh/Projects/Thesis/data/medium.csv')

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:\n", df.head(3))

Shape: (1165, 3)

Dtypes:
 grmd         int64
md_name     object
tex_text    object
dtype: object

First 3 rows:
    grmd                    md_name  \
0  1436   LIMNOCHORDIA L945 MEDIUM   
1  1479   SOLIDESULFOVIBRIO MEDIUM   
2  1481  Bold's Basal Medium (BBM)   

                                            tex_text  
0  \mono{KH$_2$PO$_4$} {0.2} {g}\n\mono{MgCl$_2$$...  
1  \mono{KH$_2$PO$_4$}                           ...  
2  \mono{Agar}{20g}\\mono{ Distilled water}{980mL...  


In [9]:
# check for missing values
missing_values = df.isnull().sum()
print("\nMissing values:\n", missing_values)


Missing values:
 grmd        0
md_name     4
tex_text    0
dtype: int64


In [10]:
# check for duplicate rows
print("Duplicate grmd:", df["grmd"].duplicated().sum())

Duplicate grmd: 0


In [11]:
# text_text length
df["tex_length"] = df["tex_text"].str.len()
print("\ntex_text length stats:\n", df["tex_length"].describe())


tex_text length stats:
 count    1165.000000
mean     1080.351931
std       818.665669
min        40.000000
25%       382.000000
50%       926.000000
75%      1592.000000
max      5756.000000
Name: tex_length, dtype: float64


In [12]:
# who are the 4 mediums with no name?
print(df[df["md_name"].isnull()])

     grmd md_name                                           tex_text  \
139  1263     NaN  \mono{NaNO$_3$}                               ...   
140  1264     NaN  \mono{Yeast extract (BD-Difco)}               ...   
144  1262     NaN  \mono{NaNO$_3$}                               ...   
648   611     NaN  \mono{Casamino acids (BD-Difco)}              ...   

     tex_length  
139         812  
140        1304  
144         813  
648         870  


In [13]:
# what does the shortest medium look like?
print(df[df["tex_length"] == df["tex_length"].min()]["tex_text"].values[0])

# anything under 100 characters
short = df[df["tex_length"] < 100]
print(f"Mediums under 100 chars: {len(short)}")
print(short[["grmd", "md_name", "tex_text"]])

\chu{Autoclaved distilled water.}
     

Mediums under 100 chars: 57
      grmd                                            md_name  \
7     1461                                 MODIFIED GAM BROTH   
9     1463                         METHANOSARCINA MFA9 MEDIUM   
20    1438                                  R2A AGAR (pH 5.5)   
42    1407                               MODIFIED YCFA MEDIUM   
54    1385                          YCFA MEDIUM WITH MANNITOL   
82    1329                                     CORN MEAL AGAR   
113   1276   MODIFIED REINFORCED CLOSTRIDIAL MEDIUM AT pH 8.0   
241   1117                 IRD MARINE DESULFOVIBRIO MEDIUM--3   
247   1110                  METHANOSARCINA SUBTERRANEA MEDIUM   
255   1087                  POREMEDIA B-CYE&#945; AGAR MEDIUM   
292   1058  DB CHARACTERIZATION MEDIUM NO.2 WITH 1\% SEA S...   
315    991                              IRD FUSIBACTER MEDIUM   
376    960                   SYNTROPHUS ACIDITROPHICUS MEDIUM   
382    949           

In [14]:
# pick one medium to test with
test_medium = df[df["grmd"] == 1205]["tex_text"].values[0]

# just print each line and what we think it is
for line in test_medium.splitlines():
    line = line.strip()
    if not line:
        continue
    
    if line.startswith(r'\mono'):
        print(f"COMPONENT  → {line}")
    elif line.startswith(r'\chu'):
        print(f"INSTRUCTION → {line}")
    elif line.startswith(r'\chuSolutionB'):
        print(f"SOLUTION B  → {line}")
    elif line.startswith(r'\chuSolutionC'):
        print(f"SOLUTION C  → {line}")
    elif line.startswith(r'\chuJCM'):
        print(f"COMMENT     → {line}")
    else:
        print(f"UNKNOWN     → {line}")

COMPONENT  → \mono{NH$_4$Cl}                                               {0.3}{g}
COMPONENT  → \mono{MgSO$_4$$\cdot$7H$_2$O}                                 {0.1}{g}
COMPONENT  → \mono{Modified Wolfe's mineral solution (see Medium No. [915])}  {5.0}{ml}
COMPONENT  → \mono{Casamino acids (BD-Difco)}                              {0.1}{g}
COMPONENT  → \mono{Yeast extract}                                          {0.02}{g}
COMPONENT  → \mono{Resazurin}                                              {0.5}{mg}
COMPONENT  → \mono{Distilled water}                                      {957.0}{ml}
INSTRUCTION → \chu{Mix components thoroughly, adjust pH to 7.0 and autoclave under a N$_2$ gas atmosphere.  After cooling, aseptically and anaerobically add the following solutions from anaerobic stocks (autoclaved or *filter-sterilized):}
COMPONENT  → \mono{0.25 M Potassium phosphate buffer (pH 7.0)}             {5.6}{ml}
COMPONENT  → \mono{Vitamin solution* (see Medium No. [915])}               {5.0}

In [15]:
test_medium = df[df["grmd"] == 1326]["tex_text"].values[0]

for line in test_medium.splitlines():
    line = line.strip()
    if not line:
        continue
    
    if line.startswith(r'\mono'):
        print(f"COMPONENT   → {line}")
    elif line.startswith(r'\chuSolutionB'):
        print(f"SOLUTION B  → {line}")
    elif line.startswith(r'\chuSolutionC'):
        print(f"SOLUTION C  → {line}")
    elif line.startswith(r'\chuJCM'):
        print(f"COMMENT     → {line}")
    elif line.startswith(r'\chu'):
        print(f"INSTRUCTION → {line}")
    else:
        print(f"UNKNOWN     → {line}")

INSTRUCTION → \chu{Solution A:}
COMPONENT   → \mono{K$_2$HPO$_4$}                                           {0.5}{g}
COMPONENT   → \mono{KH$_2$PO$_4$}                                           {0.5}{g}
COMPONENT   → \mono{NH$_4$CI}                                               {1.0}{g}
COMPONENT   → \mono{Na$_2$SO$_4$}                                           {1.0}{g}
COMPONENT   → \mono{MgSO$_4$$\cdot$7H$_2$O}                                 {2.0}{g}
COMPONENT   → \mono{CaCl$_2$$\cdot$2H$_2$O}                                 {0.1}{g}
COMPONENT   → \mono{Sodium lactate}                                         {2.0}{g}
COMPONENT   → \mono{Yeast extract}                                          {1.0}{g}
COMPONENT   → \mono{FeCl$_2$ solution (see Medium No. [187])}                 {1.0}{ml}
COMPONENT   → \mono{Trace element solution (see Medium No. [187])}            {1.0}{ml}
COMPONENT   → \mono{Trace vitamins (see Medium No. [197])}                   {10.0}{ml}
COMPONENT   → \mono{Resa

In [16]:
# how many mediums start with \chu immediately (no base components)?
no_base = df[df["tex_text"].str.strip().str.startswith(r'\chu')]["grmd"]
print(f"Mediums with no base components: {len(no_base)}")
print(no_base.values)

Mediums with no base components: 201
[1463 1438 1428 1425 1407 1385 1370 1326 1318 1295 1276 1224 1228 1204
 1209 1196 1197 1174 1100 1110 1103 1087 1089 1096 1080 1079 1067 1058
  982  986  991  995  996 1001 1003 1004 1038  972  934  938  941 1016
 1007 1008  960  949  942  910  927  926  914  873  894  895  824  834
  835  836  846  850  851  855  857  858  859  800  806  812  814  789
  775  777  770  752  753  750  726  727  731  714  640  664  712  713
  622  623  628  693  694  690  687  683  685  682  639  636  629  613
  617  602  603  591  592  594  600  585  586  577  578  579  575  576
  570  571  572  551  553  555  562  564  565  549  547  548  543  541
  519  528  530  521  517  507  503  504  502  501  499  479  456  471
  469  465  463  440  426  433  373  436  435  434  424  422  415  413
  393  394  396  398  405  362  364  367  370  380  381  374  352  330
  312  311  305  309  297  282  284  286  269  241  261  262  256  233
  218  204  205  199  200  173  178  185

In [17]:
# of the 201, how many start with \chu{Solution
sol_pattern = df[df["tex_text"].str.strip().str.startswith(r'\chu{Solution')]
print(f"Start with Solution: {len(sol_pattern)}")

# how many start with \chuSolutionB or \chuSolutionC directly?
sol_b = df[df["tex_text"].str.strip().str.startswith(r'\chuSolution')]
print(f"Start with chuSolution variant: {len(sol_b)}")

# what other \chu patterns start a medium?
no_base = df[df["tex_text"].str.strip().str.startswith(r'\chu')]
for idx, row in no_base.head(10).iterrows():
    first_line = row["tex_text"].strip().splitlines()[0]
    print(f"{row['grmd']} → {first_line[:80]}")

Start with Solution: 8
Start with chuSolution variant: 0
1463 → \chu{Use Medium No. [1462] with 21.0 g/L NaCl, instead of 9.0 g/L.}
1438 → \chu{Prepare R2A agar (see Medium No. [346]) and adjust pH to 5.5.}
1428 → \chu{Use Medium No. [815] adjusted pH to 8.0.  Reduce the amount of phosphate bu
1425 → \chu{Solution A:}
1407 → \chu{Use Medium No. [1130]. Adjust pH to 8.0.}
1385 → \chu{Use Medium No. [1130] with 0.2% mannitol. }
1370 → \chu{Prepare Medium No. [655] according to the direction, sterilize and cool to 
1326 → \chu{Solution A:}
1318 → \chu{Use Medium No. 1130 with 10% rumen fluid, clarified (see below).}
1295 → \chu{Solution A:}


In [18]:
# how many are pure references (contain "Use Medium No" or "Prepare Medium No")?
ref = df[df["tex_text"].str.contains(r'Use Medium No|Prepare Medium No')]
print(f"Reference mediums: {len(ref)}")

Reference mediums: 106


In [19]:
# mediums that start with \chu but are NOT reference and NOT solution
no_base = df[df["tex_text"].str.strip().str.startswith(r'\chu')]
reference = no_base[no_base["tex_text"].str.contains(r'Use Medium No|Prepare Medium No')]
solution = no_base[no_base["tex_text"].str.strip().str.startswith(r'\chu\{Solution')]

remaining = no_base[
    ~no_base["grmd"].isin(reference["grmd"]) & 
    ~no_base["grmd"].isin(solution["grmd"])
]

print(f"Remaining unclassified: {len(remaining)}")
print("\nFirst 10:")
for idx, row in remaining.head(10).iterrows():
    first_line = row["tex_text"].strip().splitlines()[0]
    print(f"{row['grmd']} → {first_line[:80]}")

Remaining unclassified: 102

First 10:
1438 → \chu{Prepare R2A agar (see Medium No. [346]) and adjust pH to 5.5.}
1425 → \chu{Solution A:}
1326 → \chu{Solution A:}
1295 → \chu{Solution A:}
1224 → \chu{Use JCM Medium No. [1223].  Add separately autoclaved 1.0 M glycerin soluti
1228 → \chu{Solution A:}
1204 → \chu{Solution A:}
1209 → \chu{Solution A:}
1196 → \chu{\sfi Solution A:}
1197 → \chu{\sfi Solution A:}


In [20]:
# broader reference detection
ref_pattern = r'Use Medium No|Prepare Medium No|see Medium No|Use JCM Medium No|Prepare R2A'

no_base = df[df["tex_text"].str.strip().str.startswith(r'\chu')]
reference = no_base[no_base["tex_text"].str.contains(ref_pattern)]
solution = no_base[
    no_base["tex_text"].str.contains(r'\\chu\{\\sfi Solution|\\chu\{Solution')
]

remaining = no_base[
    ~no_base["grmd"].isin(reference["grmd"]) &
    ~no_base["grmd"].isin(solution["grmd"])
]

print(f"References : {len(reference)}")
print(f"Solutions  : {len(solution)}")
print(f"Remaining  : {len(remaining)}")
print("\nRemaining first 10:")
for idx, row in remaining.head(10).iterrows():
    first_line = row["tex_text"].strip().splitlines()[0]
    print(f"{row['grmd']} → {first_line[:80]}")

References : 160
Solutions  : 55
Remaining  : 19

Remaining first 10:
1087 → \chu{Use the commercially available medium manufactured by Eiken Chemical Co., L
996 → \chu{Use the Vero E6 cell culture supernatant fluid (see below). Incubate the mi
1004 → \chu{\sfi Basic FWM solution:}
934 → \chu{Use JCM medium No. [308] supplemented with 0.1 g/L (final) yeast extract an
664 → \chu{Autoclaved distilled water.}
712 → \chu{Use commercially available Lowenstein--Jensen medium slants (BD 220908).}
690 → \chu{Use commercially available product [Anaero Columbia agar with rabbit blood 
586 → \chu{Prepare Columbia blood agar base (Oxoid CM331) according to the direction,
507 → \chu{Use Solution A of Medium No. [284] supplemented with 0.1 g sodium acetate a
435 → \chu{Use Solution A of Medium No. [284] supplemented with final 1.0 ml Trace vit


In [21]:
# print ALL 19 remaining first lines
for idx, row in remaining.iterrows():
    first_line = row["tex_text"].strip().splitlines()[0]
    print(f"{row['grmd']} → {first_line[:90]}")
    

1087 → \chu{Use the commercially available medium manufactured by Eiken Chemical Co., Ltd.}
996 → \chu{Use the Vero E6 cell culture supernatant fluid (see below). Incubate the microorganis
1004 → \chu{\sfi Basic FWM solution:}
934 → \chu{Use JCM medium No. [308] supplemented with 0.1 g/L (final) yeast extract and
664 → \chu{Autoclaved distilled water.}
712 → \chu{Use commercially available Lowenstein--Jensen medium slants (BD 220908).}
690 → \chu{Use commercially available product [Anaero Columbia agar with rabbit blood (BD--BBL)]
586 → \chu{Prepare Columbia blood agar base (Oxoid CM331) according to the direction,
507 → \chu{Use Solution A of Medium No. [284] supplemented with 0.1 g sodium acetate and
435 → \chu{Use Solution A of Medium No. [284] supplemented with final 1.0 ml Trace vitamins
312 → \chu{Use the Medium No. [310] without KSCN solution. Aseptically add 20.0
311 → \chu{Use the Medium No. [310] without KSCN solution. Aseptically add 20.0
282 → \chu{Prepare Columbia blood ag

In [7]:
import re
tags = df["tex_text"].str.findall(r'\\[a-zA-Z]+')
unique_tags = set(tag for row in tags for tag in row)
print(unique_tags)


{'\\mono', '\\mu', '\\chuSolutionC', '\\chuJCM', '\\alpha', '\\hspace', '\\beta', '\\ge', '\\sfi', '\\chuSolutionB', '\\cdot', '\\chu', '\\Mix', '\\HCl'}
